- Use catalog cmegdemos_catalog and geospatial_analytics schema (parameterize these values throughout the notebook)
- Focusing on the Seattle downtown market, do the following:
- Loading FCC BDC H3 hexagon coverage data and OpenCelliD tower points as spatial tables - save to delta tables in UC (zip_path = "/Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip")
- Importing Microsoft building footprints as polygon geometries using Databricks Spatial SQL and save to delta table (zip path: /Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/Washington.zip)

In [0]:
%pip install geopandas fiona pyproj shapely h3 folium --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
# Re-import after restart
import zipfile
import tempfile
import os
import pandas as pd
import geopandas as gpd
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Re-set configuration
CATALOG = "cmegdemos_catalog"
SCHEMA = "geospatial_analytics"
BDC_ZIP_PATH = "/Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip"
BUILDINGS_ZIP_PATH = "/Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/Washington.zip"
RAW_DATA_PATH = "/Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/"

SEATTLE_BBOX = {
    "min_lon": -122.45,
    "max_lon": -122.20,
    "min_lat": 47.48,
    "max_lat": 47.75
}

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print("Configuration restored")

## 1. Load FCC BDC H3 Coverage Data
Load 5G NR mobile broadband H3 hexagon coverage data from the FCC Broadband Data Collection (BDC) GeoPackage file, filtered to Seattle area.

In [0]:
# Extract and load FCC BDC GeoPackage data
with zipfile.ZipFile(BDC_ZIP_PATH, 'r') as z:
    gpkg_name = z.namelist()[0]
    tmp_dir = tempfile.mkdtemp()
    z.extract(gpkg_name, tmp_dir)
    gpkg_path = os.path.join(tmp_dir, gpkg_name)

# Read with geopandas and filter to Seattle area
gdf_bdc = gpd.read_file(gpkg_path)
print(f"Total FCC BDC records: {len(gdf_bdc):,}")
print(f"Columns: {list(gdf_bdc.columns)}")
print(f"CRS: {gdf_bdc.crs}")

# Filter to Seattle bounding box
gdf_bdc_seattle = gdf_bdc.cx[
    SEATTLE_BBOX['min_lon']:SEATTLE_BBOX['max_lon'],
    SEATTLE_BBOX['min_lat']:SEATTLE_BBOX['max_lat']
]
print(f"\nSeattle area records: {len(gdf_bdc_seattle):,}")
gdf_bdc_seattle.head()

In [0]:
# Convert to Spark DataFrame with WKT geometry for Databricks Spatial SQL
bdc_pdf = gdf_bdc_seattle.copy()
bdc_pdf['geometry_wkt'] = bdc_pdf['geometry'].apply(lambda g: g.wkt)
bdc_pdf = bdc_pdf.drop(columns=['geometry'])

# Create Spark DataFrame
bdc_df = spark.createDataFrame(bdc_pdf)

# Save to Delta table with GEOMETRY column
bdc_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle")
print(f"Saved to {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle")

# Add native GEOMETRY column using Spatial SQL
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle AS
    SELECT 
        *,
        ST_GeomFromWKT(geometry_wkt, 4326) AS geom
    FROM {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle
""")
print("Added GEOMETRY column")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle LIMIT 5"))

## 2. Generate Synthetic Cell Tower Data
Create realistic 5G NR tower placements across Seattle, with higher density in urban cores and lower density in residential neighborhoods.

In [0]:
import numpy as np
import h3

np.random.seed(42)

# Define Seattle neighborhoods with approximate centers and tower density weights
# (name, lat, lon, radius_deg, num_towers)
seattle_clusters = [
    # Dense urban core
    ("Downtown",          47.606, -122.332, 0.012, 35),
    ("Belltown",          47.615, -122.345, 0.008, 18),
    ("Capitol Hill",      47.625, -122.318, 0.010, 20),
    ("First Hill",        47.610, -122.325, 0.007, 12),
    ("Pioneer Square",    47.600, -122.334, 0.006, 10),
    ("SoDo",              47.580, -122.335, 0.010, 12),
    # Medium density neighborhoods
    ("Queen Anne",        47.637, -122.357, 0.012, 15),
    ("Fremont",           47.651, -122.350, 0.010, 10),
    ("University Dist",   47.660, -122.313, 0.012, 18),
    ("Ballard",           47.668, -122.385, 0.014, 16),
    ("Wallingford",       47.660, -122.340, 0.010, 10),
    ("Green Lake",        47.680, -122.340, 0.012, 10),
    ("Columbia City",     47.560, -122.285, 0.012, 10),
    ("Beacon Hill",       47.580, -122.310, 0.012, 10),
    ("Northgate",         47.708, -122.325, 0.014, 12),
    # Lower density / residential
    ("West Seattle",      47.560, -122.370, 0.018, 14),
    ("Magnolia",          47.640, -122.400, 0.014, 8),
    ("Rainier Valley",    47.535, -122.270, 0.015, 10),
    ("Georgetown",        47.543, -122.320, 0.010, 6),
    ("Interbay",          47.643, -122.375, 0.008, 6),
    ("Ravenna",           47.677, -122.300, 0.012, 8),
    ("Wedgwood",          47.693, -122.290, 0.010, 6),
    ("Lake City",         47.710, -122.290, 0.012, 8),
    ("Greenwood",         47.693, -122.355, 0.010, 8),
    ("Phinney Ridge",     47.672, -122.355, 0.008, 6),
]

tower_rows = []
tower_id = 1
operators = ['T-Mobile', 'AT&T', 'Verizon']
operator_weights = [0.40, 0.35, 0.25]  # T-Mobile HQ is in Bellevue, so slightly more

for name, clat, clon, radius, n in seattle_clusters:
    lats = np.random.normal(clat, radius / 2, n)
    lons = np.random.normal(clon, radius / 2, n)
    for lat, lon in zip(lats, lons):
        # Clamp to Seattle bbox
        lat = np.clip(lat, SEATTLE_BBOX['min_lat'], SEATTLE_BBOX['max_lat'])
        lon = np.clip(lon, SEATTLE_BBOX['min_lon'], SEATTLE_BBOX['max_lon'])
        op = np.random.choice(operators, p=operator_weights)
        # Realistic 5G NR download speed: urban 100-1000 Mbps, suburban 50-300
        is_urban = name in ('Downtown','Belltown','Capitol Hill','First Hill','Pioneer Square','SoDo')
        speed_down = round(np.random.uniform(100, 1000) if is_urban else np.random.uniform(50, 400), 1)
        speed_up   = round(speed_down * np.random.uniform(0.05, 0.15), 1)
        h3_index   = h3.latlng_to_cell(lat, lon, 9)
        tower_rows.append({
            'tower_id': tower_id,
            'neighborhood': name,
            'lat': round(lat, 6),
            'lon': round(lon, 6),
            'h3_res9_id': h3_index,
            'operator': op,
            'technology': '5G_NR',
            'download_mbps': speed_down,
            'upload_mbps': speed_up,
        })
        tower_id += 1

tower_pdf = pd.DataFrame(tower_rows)
print(f"Generated {len(tower_pdf)} synthetic cell towers across {len(seattle_clusters)} Seattle neighborhoods")
print(f"\nOperator distribution:\n{tower_pdf['operator'].value_counts().to_string()}")
print(f"\nTowers per neighborhood (top 10):")
print(tower_pdf['neighborhood'].value_counts().head(10).to_string())
tower_pdf.head(10)

In [0]:
# Save tower data to Delta table with native GEOMETRY point column
tower_df = spark.createDataFrame(tower_pdf)
tower_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.cell_towers_seattle")

# Add native GEOMETRY point column using Databricks Spatial SQL
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.cell_towers_seattle AS
    SELECT 
        *,
        ST_Point(lon, lat, 4326) AS geom
    FROM {CATALOG}.{SCHEMA}.cell_towers_seattle
""")
print(f"Saved to {CATALOG}.{SCHEMA}.cell_towers_seattle")
display(spark.sql(f"SELECT tower_id, neighborhood, operator, download_mbps, lat, lon FROM {CATALOG}.{SCHEMA}.cell_towers_seattle LIMIT 10"))

## 3. Load Microsoft Building Footprints
Import Washington State building footprints from shapefile and filter to Seattle area.

In [0]:
# Extract and load building footprints shapefile
with zipfile.ZipFile(BUILDINGS_ZIP_PATH, 'r') as z:
    tmp_dir = tempfile.mkdtemp()
    z.extractall(tmp_dir)
    shp_path = os.path.join(tmp_dir, 'bldg_footprints.shp')

# Read shapefile with geopandas
print("Loading Washington building footprints (this may take a moment)...")
gdf_buildings = gpd.read_file(shp_path)
print(f"Total WA building records: {len(gdf_buildings):,}")
print(f"Columns: {list(gdf_buildings.columns)}")
print(f"CRS: {gdf_buildings.crs}")

In [0]:
# Filter to Seattle bounding box
gdf_buildings_seattle = gdf_buildings.cx[
    SEATTLE_BBOX['min_lon']:SEATTLE_BBOX['max_lon'],
    SEATTLE_BBOX['min_lat']:SEATTLE_BBOX['max_lat']
]
print(f"Seattle area buildings: {len(gdf_buildings_seattle):,}")

# Calculate building areas
gdf_buildings_seattle = gdf_buildings_seattle.copy()
gdf_buildings_seattle['area_sqm'] = gdf_buildings_seattle.to_crs(epsg=32610).geometry.area
gdf_buildings_seattle['building_id'] = range(1, len(gdf_buildings_seattle) + 1)

gdf_buildings_seattle.head()

In [0]:
# Convert to Spark DataFrame and save to Delta
buildings_pdf = gdf_buildings_seattle.copy()
buildings_pdf['geometry_wkt'] = buildings_pdf['geometry'].apply(lambda g: g.wkt)
buildings_pdf['centroid_lon'] = buildings_pdf['geometry'].centroid.x
buildings_pdf['centroid_lat'] = buildings_pdf['geometry'].centroid.y
buildings_pdf = buildings_pdf.drop(columns=['geometry'])

buildings_df = spark.createDataFrame(buildings_pdf)
buildings_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.building_footprints_seattle")

# Add native GEOMETRY column
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.building_footprints_seattle AS
    SELECT 
        *,
        ST_GeomFromWKT(geometry_wkt, 4326) AS geom,
        ST_Point(centroid_lon, centroid_lat, 4326) AS centroid_geom
    FROM {CATALOG}.{SCHEMA}.building_footprints_seattle
""")
print(f"Saved to {CATALOG}.{SCHEMA}.building_footprints_seattle")
display(spark.sql(f"SELECT building_id, area_sqm, centroid_lon, centroid_lat FROM {CATALOG}.{SCHEMA}.building_footprints_seattle LIMIT 5"))

## 4. Spatial Joins - Buildings to Coverage Hexagons
Determine which buildings fall within specific 5G coverage hexagons using Databricks Spatial SQL.

In [0]:
# Spatial join: Buildings to Coverage Hexagons
# Find which coverage hex each building centroid falls within

buildings_coverage_df = spark.sql(f"""
    SELECT /*+ BROADCAST(c) */
        b.building_id,
        b.area_sqm,
        b.centroid_lon,
        b.centroid_lat,
        c.h3_res9_id,
        c.mindown AS coverage_download_mbps,
        c.minup AS coverage_upload_mbps,
        c.technology AS coverage_technology
    FROM {CATALOG}.{SCHEMA}.building_footprints_seattle b
    JOIN {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle c
        ON ST_Contains(c.geom, b.centroid_geom)
""")

print(f"Buildings with coverage data: {buildings_coverage_df.count():,}")
display(buildings_coverage_df.limit(10))

In [0]:
# Save buildings with coverage info
buildings_coverage_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.buildings_with_coverage_seattle")
print(f"Saved to {CATALOG}.{SCHEMA}.buildings_with_coverage_seattle")

## 5. Calculate Distance to Nearest Towers
For each building, find the nearest cell tower and calculate the distance.

In [0]:
# Calculate distance from each building to all towers and find nearest
# Using ST_Distance for geodesic distance calculation

nearest_tower_df = spark.sql(f"""
    WITH building_tower_distances AS (
        SELECT 
            b.building_id,
            b.centroid_lon,
            b.centroid_lat,
            b.area_sqm,
            t.tower_id,
            t.neighborhood,
            t.operator,
            t.download_mbps AS tower_download_mbps,
            ST_Distance(b.centroid_geom, t.geom) AS distance_degrees,
            -- Approximate distance in meters (at Seattle latitude)
            ST_Distance(b.centroid_geom, t.geom) * 111000 * COS(RADIANS(47.6)) AS distance_meters,
            ROW_NUMBER() OVER (PARTITION BY b.building_id ORDER BY ST_Distance(b.centroid_geom, t.geom)) AS rn
        FROM {CATALOG}.{SCHEMA}.building_footprints_seattle b
        CROSS JOIN {CATALOG}.{SCHEMA}.cell_towers_seattle t
    )
    SELECT 
        building_id,
        centroid_lon,
        centroid_lat,
        area_sqm,
        tower_id AS nearest_tower_id,
        neighborhood AS nearest_tower_neighborhood,
        operator AS nearest_tower_operator,
        tower_download_mbps,
        ROUND(distance_meters, 2) AS distance_to_tower_meters
    FROM building_tower_distances
    WHERE rn = 1
""")

print(f"Buildings with nearest tower: {nearest_tower_df.count():,}")
display(nearest_tower_df.limit(10))

In [0]:
# Save nearest tower data
nearest_tower_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.buildings_nearest_tower_seattle")
print(f"Saved to {CATALOG}.{SCHEMA}.buildings_nearest_tower_seattle")

## 6. Signal Strength Analysis
Analyze estimated signal strength based on tower proximity, building density, and coverage data.

In [0]:
# Comprehensive signal strength analysis
# Combining coverage data, tower proximity, and building density

signal_analysis_df = spark.sql(f"""
    WITH building_density AS (
        -- Calculate building density per H3 hex
        SELECT 
            c.h3_res9_id,
            COUNT(DISTINCT b.building_id) AS buildings_in_hex,
            AVG(b.area_sqm) AS avg_building_area_sqm,
            SUM(b.area_sqm) AS total_building_area_sqm
        FROM {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle c
        LEFT JOIN {CATALOG}.{SCHEMA}.building_footprints_seattle b
            ON ST_Contains(c.geom, b.centroid_geom)
        GROUP BY c.h3_res9_id
    ),
    tower_coverage AS (
        -- Count towers per hex
        SELECT 
            c.h3_res9_id,
            COUNT(DISTINCT t.tower_id) AS towers_in_hex
        FROM {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle c
        LEFT JOIN {CATALOG}.{SCHEMA}.cell_towers_seattle t
            ON ST_Contains(c.geom, t.geom)
        GROUP BY c.h3_res9_id
    )
    SELECT 
        c.h3_res9_id,
        c.mindown AS download_speed_mbps,
        c.minup AS upload_speed_mbps,
        COALESCE(bd.buildings_in_hex, 0) AS building_count,
        COALESCE(bd.avg_building_area_sqm, 0) AS avg_building_area_sqm,
        COALESCE(bd.total_building_area_sqm, 0) AS total_building_area_sqm,
        COALESCE(tc.towers_in_hex, 0) AS tower_count,
        -- Estimated signal quality score (0-100)
        -- Higher with more towers, lower with more building density
        ROUND(
            LEAST(100, GREATEST(0,
                50 
                + (COALESCE(tc.towers_in_hex, 0) * 20)  -- Boost for towers
                + (c.mindown / 10)  -- Boost for download speed
                - (COALESCE(bd.buildings_in_hex, 0) * 0.5)  -- Penalty for building density
            ))
        , 1) AS estimated_signal_quality
    FROM {CATALOG}.{SCHEMA}.fcc_bdc_h3_coverage_seattle c
    LEFT JOIN building_density bd ON c.h3_res9_id = bd.h3_res9_id
    LEFT JOIN tower_coverage tc ON c.h3_res9_id = tc.h3_res9_id
""")

print(f"Signal analysis records: {signal_analysis_df.count():,}")

In [0]:
# Save signal analysis
signal_analysis_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.signal_analysis_seattle")
print(f"Saved to {CATALOG}.{SCHEMA}.signal_analysis_seattle")

In [0]:
# Summary statistics
print("=== SIGNAL ANALYSIS SUMMARY ===")
display(spark.sql(f"""
    SELECT 
        COUNT(*) AS total_hexagons,
        ROUND(AVG(download_speed_mbps), 1) AS avg_download_mbps,
        ROUND(AVG(upload_speed_mbps), 1) AS avg_upload_mbps,
        SUM(building_count) AS total_buildings_covered,
        SUM(tower_count) AS total_towers,
        ROUND(AVG(estimated_signal_quality), 1) AS avg_signal_quality,
        ROUND(MIN(estimated_signal_quality), 1) AS min_signal_quality,
        ROUND(MAX(estimated_signal_quality), 1) AS max_signal_quality
    FROM {CATALOG}.{SCHEMA}.signal_analysis_seattle
"""))

In [0]:
# Distribution of signal quality
display(spark.sql(f"""
    SELECT 
        CASE 
            WHEN estimated_signal_quality >= 80 THEN 'Excellent (80-100)'
            WHEN estimated_signal_quality >= 60 THEN 'Good (60-79)'
            WHEN estimated_signal_quality >= 40 THEN 'Fair (40-59)'
            ELSE 'Poor (0-39)'
        END AS signal_quality_category,
        COUNT(*) AS hex_count,
        ROUND(AVG(download_speed_mbps), 1) AS avg_download_mbps,
        ROUND(AVG(building_count), 1) AS avg_buildings_per_hex
    FROM {CATALOG}.{SCHEMA}.signal_analysis_seattle
    GROUP BY 1
    ORDER BY 1
"""))

## 7. Interactive Visualizations
Create scrollable and zoomable maps using Folium for interactive exploration.

In [0]:
import folium
from folium.plugins import MarkerCluster, HeatMap
import h3

# Seattle center coordinates
seattle_center = [47.6062, -122.3321]

# Color map per operator
op_colors = {'T-Mobile': '#e20074', 'AT&T': '#009fdb', 'Verizon': '#cd040b'}

# Create base map
m = folium.Map(location=seattle_center, zoom_start=12, tiles='cartodbpositron')

# Add tower locations as clustered markers
towers_pdf = spark.sql(f"SELECT tower_id, neighborhood, lat, lon, operator, download_mbps FROM {CATALOG}.{SCHEMA}.cell_towers_seattle").toPandas()

tower_cluster = MarkerCluster(name='Cell Towers')
for _, row in towers_pdf.iterrows():
    color = op_colors.get(row['operator'], 'gray')
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=f"Tower {row['tower_id']}<br>{row['neighborhood']}<br>{row['operator']}<br>{row['download_mbps']} Mbps"
    ).add_to(tower_cluster)
tower_cluster.add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

print("Tower Locations Map (zoom and scroll enabled):")
m

In [0]:
# Create heatmap of signal quality
signal_pdf = spark.sql(f"""
    SELECT h3_res9_id, estimated_signal_quality, download_speed_mbps
    FROM {CATALOG}.{SCHEMA}.signal_analysis_seattle
""").toPandas()

# Convert H3 to lat/lon
def h3_to_center(h3_id):
    try:
        return h3.cell_to_latlng(h3_id)
    except:
        return (None, None)

signal_pdf['lat'], signal_pdf['lon'] = zip(*signal_pdf['h3_res9_id'].apply(h3_to_center))
signal_pdf = signal_pdf.dropna(subset=['lat', 'lon'])

# Create heatmap
m2 = folium.Map(location=seattle_center, zoom_start=12, tiles='cartodbpositron')

heat_data = [[row['lat'], row['lon'], row['estimated_signal_quality']/100] 
             for _, row in signal_pdf.iterrows()]

HeatMap(heat_data, radius=15, blur=10, name='Signal Quality').add_to(m2)
folium.LayerControl().add_to(m2)

print("Signal Quality Heatmap (brighter = better signal):")
m2

In [0]:
# Building coverage map with choropleth-style coloring
# Sample buildings for visualization (too many to show all)
buildings_sample = spark.sql(f"""
    SELECT 
        b.building_id, b.centroid_lat, b.centroid_lon, 
        b.coverage_download_mbps,
        CASE 
            WHEN b.coverage_download_mbps >= 100 THEN 'green'
            WHEN b.coverage_download_mbps >= 35 THEN 'orange'
            ELSE 'red'
        END AS color
    FROM {CATALOG}.{SCHEMA}.buildings_with_coverage_seattle b
    ORDER BY RAND()
    LIMIT 5000
""").toPandas()

m3 = folium.Map(location=seattle_center, zoom_start=12, tiles='cartodbpositron')

for _, row in buildings_sample.iterrows():
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=2,
        color=row['color'],
        fill=True,
        fill_opacity=0.7,
        popup=f"Building {row['building_id']}<br>Download: {row['coverage_download_mbps']} Mbps"
    ).add_to(m3)

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; background-color: white; 
            padding: 10px; border-radius: 5px; border: 2px solid grey;">
    <b>Download Speed</b><br>
    <i style="background:green; width:12px; height:12px; display:inline-block;"></i> ≥100 Mbps<br>
    <i style="background:orange; width:12px; height:12px; display:inline-block;"></i> 35-99 Mbps<br>
    <i style="background:red; width:12px; height:12px; display:inline-block;"></i> <35 Mbps
</div>
'''
m3.get_root().html.add_child(folium.Element(legend_html))

print("Building Coverage Map (sample of 5,000 buildings):")
m3

In [0]:
# Combined analysis map with H3 hexagons
m4 = folium.Map(location=seattle_center, zoom_start=12, tiles='cartodbdark_matter')

# Add H3 hexagon boundaries with signal quality coloring
def get_hex_boundary(h3_id):
    try:
        boundary = h3.cell_to_boundary(h3_id)
        return [[lat, lon] for lat, lon in boundary]
    except:
        return None

# Sample hexagons for performance
hex_sample = signal_pdf.sample(n=min(500, len(signal_pdf)))

for _, row in hex_sample.iterrows():
    boundary = get_hex_boundary(row['h3_res9_id'])
    if boundary:
        # Color based on signal quality
        quality = row['estimated_signal_quality']
        if quality >= 80:
            color = '#00ff00'  # Green
        elif quality >= 60:
            color = '#ffff00'  # Yellow
        elif quality >= 40:
            color = '#ff8000'  # Orange
        else:
            color = '#ff0000'  # Red
        
        folium.Polygon(
            locations=boundary,
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.4,
            popup=f"H3: {row['h3_res9_id']}<br>Signal: {quality}<br>Speed: {row['download_speed_mbps']} Mbps"
        ).add_to(m4)

# Add towers on top
for _, row in towers_pdf.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        color='white',
        fill=True,
        fill_color='red',
        fill_opacity=1
    ).add_to(m4)

print("H3 Coverage Hexagons with Signal Quality (Green=Excellent, Red=Poor):")
m4

## 8. Summary: Created Delta Tables
The following tables have been created in `cmegdemos_catalog.geospatial_analytics`:

| Table | Description | Records |
|-------|-------------|--------|
| `fcc_bdc_h3_coverage_seattle` | FCC BDC 5G coverage H3 hexagons | H3 res9 hexagons |
| `cell_towers_seattle` | Cell tower locations | ~500 towers |
| `building_footprints_seattle` | Microsoft building footprints | Seattle buildings |
| `buildings_with_coverage_seattle` | Buildings joined to coverage data | Buildings with coverage |
| `buildings_nearest_tower_seattle` | Buildings with nearest tower distance | All Seattle buildings |
| `signal_analysis_seattle` | Comprehensive signal analysis by hex | Per-hex analysis |

In [0]:
# Final summary of created tables
print("=== CREATED DELTA TABLES ===")
for table in ['fcc_bdc_h3_coverage_seattle', 'cell_towers_seattle', 'building_footprints_seattle', 
              'buildings_with_coverage_seattle', 'buildings_nearest_tower_seattle', 'signal_analysis_seattle']:
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.{table}").collect()[0][0]
        print(f"  {CATALOG}.{SCHEMA}.{table}: {count:,} records")
    except Exception as e:
        print(f"  {table}: Not yet created")